# Prediksi Nilai Siswa dengan Decision Tree
### Seri Praktik Data Science - B5 | Menata Data Lab

---

Notebook pendamping ebook **"Prediksi Nilai Siswa dengan Decision Tree"**.

| Aspek | Detail |
|-------|--------|
| **Model** | Decision Tree Classifier |
| **Bidang** | Pendidikan |
| **Dataset** | UCI Student Performance (395 siswa, 32 variabel) |
| **Target** | Lulus (G3 >= 10) vs Gagal (G3 < 10) |
| **Bahasa** | Python + scikit-learn |

**Alur notebook ini:**
1. Pengantar & Instalasi
2. Teori Decision Tree (Gini & Entropy)
3. Mengenal & Memuat Dataset
4. Preprocessing Data
5. Membangun Pohon Keputusan (Baseline)
6. Pruning & Hyperparameter Tuning
7. Evaluasi & Visualisasi
8. Interpretasi Aturan untuk Intervensi
9. Ekspor Model (.pkl)
10. Uji Deployment dengan Data Baru (dari model yang diekspor)

## 0. Instalasi & Import Library

In [1]:
# Jalankan di Google Colab atau lokal -- install jika belum ada
#!pip install pandas scikit-learn matplotlib seaborn ucimlrepo

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split, GridSearchCV, learning_curve
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

print("Library berhasil dimuat!")

---
## Bab 1 -- Pengantar: Prediktif Analitik dalam Dunia Pendidikan

Setiap semester, ribuan siswa diam-diam meluncur menuju kegagalan akademik -- dan seringkali baru disadari ketika nilai akhir sudah keluar.

**Prediktif analitik pendidikan** mengubah arsip data pasif (nilai, kehadiran, latar belakang) menjadi sistem yang menjawab:
> *"Siswa mana yang paling berisiko gagal, dan apa faktornya?"*

### Mengapa Decision Tree?
- **Dapat dijelaskan** -- menghasilkan aturan JIKA-MAKA yang bisa dibaca guru, kepala sekolah, bahkan orang tua.
- **Tidak memerlukan scaling** -- pohon hanya membandingkan nilai dengan ambang batas.
- Contoh aturan: *"JIKA nilai periode 2 <= 9.5 DAN riwayat gagal kelas > 0, MAKA siswa berisiko tinggi tidak lulus."*

### Studi Kasus
Membangun **sistem peringatan dini kelulusan** dari data 395 siswa sekolah menengah di Portugal.

---
## Bab 2 -- Memahami Cara Kerja Decision Tree

### Anatomi Pohon Keputusan

| Istilah | Makna |
|---------|-------|
| **Root Node** | Titik awal -- split pertama atas seluruh data |
| **Internal Node** | Titik keputusan berisi satu kondisi |
| **Branch** | Hasil dari kondisi: benar (kiri) atau salah (kanan) |
| **Leaf Node** | Titik akhir -- berisi prediksi kelas |
| **Depth** | Jumlah tingkat dari akar ke daun terjauh |

### Gini Impurity
$$\text{Gini} = 1 - \sum (p_i)^2$$

Mengukur seberapa sering sampel akan salah diklasifikasikan jika dilabeli secara acak. Nilai 0 = murni, 0.5 = paling tidak murni.

### Entropy & Information Gain
$$\text{Entropy} = -\sum p_i \cdot \log_2(p_i)$$
$$\text{Information Gain} = \text{Entropy}_{\text{induk}} - \sum (\text{bobot} \times \text{Entropy}_{\text{anak}})$$

Pohon memilih split dengan information gain terbesar.

In [3]:
# === Visualisasi: Kurva Gini vs Entropy (Gambar 2.1 di ebook) ===

p = np.linspace(0.001, 0.999, 500)
gini = 2 * p * (1 - p)
entropy = -(p * np.log2(p) + (1 - p) * np.log2(1 - p))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(p, gini, label='Gini Impurity', linewidth=2)
ax.plot(p, entropy, label='Entropy', linewidth=2, linestyle='--')
ax.axvline(0.5, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Proporsi kelas positif (p)')
ax.set_ylabel('Nilai ketidakmurnian')
ax.set_title('Gambar 2.1 -- Kurva Gini Impurity vs Entropy')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Kedua kurva mencapai puncak saat p=0.5 (campuran 50:50)")
print("dan turun ke 0 saat satu kelas mendominasi penuh.")

Kedua kurva mencapai puncak saat p=0.5 (campuran 50:50)
dan turun ke 0 saat satu kelas mendominasi penuh.


In [4]:
# === Contoh Perhitungan Manual Gini ===
# Node berisi 8 siswa Gagal dan 2 siswa Lulus

p_gagal = 8 / 10
p_lulus = 2 / 10
gini_val = 1 - (p_gagal**2 + p_lulus**2)

print(f"p(Gagal) = {p_gagal}")
print(f"p(Lulus) = {p_lulus}")
print(f"Gini = 1 - ({p_gagal}² + {p_lulus}²) = 1 - ({p_gagal**2} + {p_lulus**2}) = {gini_val:.2f}")
print(f"\nNode ini cukup murni -- didominasi kelas Gagal.")

p(Gagal) = 0.8
p(Lulus) = 0.2
Gini = 1 - (0.8² + 0.2²) = 1 - (0.6400000000000001 + 0.04000000000000001) = 0.32

Node ini cukup murni -- didominasi kelas Gagal.


---
## Bab 3 -- Mengenal Dataset Student Performance

Dataset dikumpulkan oleh **Cortez & Silva (2008)** dari dua sekolah menengah di Portugal.
- **395 siswa**, **32 variabel** prediktor
- Kombinasi faktor akademik dan latar belakang sosial
- Subset mata pelajaran **Matematika** (`student-mat.csv`)

### Definisi Target
```python
lulus = 1 if G3 >= 10 else 0  # 1 = Lulus, 0 = Gagal
```
Hasil: 265 siswa (67.1%) Lulus, 130 siswa (32.9%) Gagal.

In [5]:
# === Mengunduh & Memuat Dataset ===
# Dataset: UCI Student Performance -- subset Matematika (student-mat.csv, 395 siswa)

import zipfile, io, urllib.request

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00320/student.zip"
resp = urllib.request.urlopen(url)
z = zipfile.ZipFile(io.BytesIO(resp.read()))
with z.open('student-mat.csv') as f:
    df = pd.read_csv(f, sep=';')
print("Dataset dimuat dari UCI Repository (student-mat.csv).")

# Membuat variabel target biner dari nilai akhir G3
df["lulus"] = (df["G3"] >= 10).astype(int)

print(f"Dimensi data : {df.shape}")
print(f"\nDistribusi target:")
print(df["lulus"].value_counts())

Dataset dimuat dari UCI Repository (student-mat.csv).
Dimensi data : (395, 34)

Distribusi target:
lulus
1    265
0    130
Name: count, dtype: int64


In [6]:
# === Eksplorasi Awal (EDA) ===

# Tipe data & jumlah non-null tiap kolom
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 34 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   school      395 non-null    str  
 1   sex         395 non-null    str  
 2   age         395 non-null    int64
 3   address     395 non-null    str  
 4   famsize     395 non-null    str  
 5   Pstatus     395 non-null    str  
 6   Medu        395 non-null    int64
 7   Fedu        395 non-null    int64
 8   Mjob        395 non-null    str  
 9   Fjob        395 non-null    str  
 10  reason      395 non-null    str  
 11  guardian    395 non-null    str  
 12  traveltime  395 non-null    int64
 13  studytime   395 non-null    int64
 14  failures    395 non-null    int64
 15  schoolsup   395 non-null    str  
 16  famsup      395 non-null    str  
 17  paid        395 non-null    str  
 18  activities  395 non-null    str  
 19  nursery     395 non-null    str  
 20  higher      395 non-null    str  
 21  inte

In [7]:
# Ringkasan statistik kolom numerik
df.describe()

,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3,lulus
count,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000
mean,16.696203,2.749367,2.521519,1.448101,2.035443,0.334177,3.944304,3.235443,3.108861,1.481013,2.291139,3.554430,5.708861,10.908861,10.713924,10.415190,0.670886
std,1.276043,1.094735,1.088201,0.697505,0.839240,0.743651,0.896659,0.998862,1.113278,0.890741,1.287897,1.390303,8.003096,3.319195,3.761505,4.581443,0.470487
min,15.000000,0.000000,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,3.000000,0.000000,0.000000,0.000000
25%,16.000000,2.000000,2.000000,1.000000,1.000000,0.000000,4.000000,3.000000,2.000000,1.000000,1.000000,3.000000,0.000000,8.000000,9.000000,8.000000,0.000000
50%,17.000000,3.000000,2.000000,1.000000,2.000000,0.000000,4.000000,3.000000,3.000000,1.000000,2.000000,4.000000,4.000000,11.000000,11.000000,11.000000,1.000000
75%,18.000000,4.000000,3.000000,2.000000,2.000000,0.000000,5.000000,4.000000,4.000000,2.000000,3.000000,5.000000,8.000000,13.000000,13.000000,14.000000,1.000000
max,22.000000,4.000000,4.000000,4.000000,4.000000,3.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,75.000000,19.000000,19.000000,20.000000,1.000000


In [8]:
# Menghitung nilai hilang per kolom
missing = df.isnull().sum()
print("Nilai hilang per kolom:")
print(missing[missing > 0] if missing.sum() > 0 else "Tidak ada nilai hilang -- dataset bersih!")

Nilai hilang per kolom:
Tidak ada nilai hilang -- dataset bersih!


In [9]:
# === Distribusi Kelas Target (Gambar 3.1 di ebook) ===

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
counts = df["lulus"].value_counts()
labels = ["Gagal (0)", "Lulus (1)"]
colors = ["#e74c3c", "#2ecc71"]
axes[0].bar(labels, [counts[0], counts[1]], color=colors, edgecolor='black')
axes[0].set_ylabel("Jumlah Siswa")
axes[0].set_title("Distribusi Kelas Target")
for i, v in enumerate([counts[0], counts[1]]):
    axes[0].text(i, v + 3, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie([counts[0], counts[1]], labels=labels, colors=colors,
            autopct='%1.1f%%', startangle=90, explode=(0.05, 0))
axes[1].set_title("Proporsi Lulus vs Gagal")

plt.suptitle("Gambar 3.1 -- Distribusi Kelas Target", fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Dengan {counts[0]} siswa Gagal ({counts[0]/len(df)*100:.1f}%) dan "
      f"{counts[1]} siswa Lulus ({counts[1]/len(df)*100:.1f}%), distribusi cukup seimbang.")

Dengan 130 siswa Gagal (32.9%) dan 265 siswa Lulus (67.1%), distribusi cukup seimbang.


In [10]:
# === Variabel Penting dalam Dataset ===

variabel_info = pd.DataFrame({
    'Variabel': ['G1, G2', 'failures', 'studytime', 'absences', 'goout',
                 'Walc, Dalc', 'Medu, Fedu', 'schoolsup', 'higher', 'Mjob, Fjob'],
    'Jenis': ['Numerik (0-20)', 'Numerik (0-3)', 'Ordinal (1-4)', 'Numerik',
              'Ordinal (1-5)', 'Ordinal (1-5)', 'Ordinal (0-4)', 'Biner', 'Biner', 'Nominal'],
    'Makna': ['Nilai periode 1 & 2', 'Riwayat gagal kelas', 'Waktu belajar mingguan',
              'Jumlah ketidakhadiran', 'Frekuensi keluar dengan teman',
              'Konsumsi alkohol akhir pekan & hari biasa', 'Pendidikan ibu & ayah',
              'Bimbingan belajar dari sekolah', 'Berniat ke perguruan tinggi',
              'Pekerjaan ibu & ayah']
})
variabel_info

,Variabel,Jenis,Makna
0,"G1, G2",Numerik (0-20),Nilai periode 1 & 2
1,failures,Numerik (0-3),Riwayat gagal kelas
2,studytime,Ordinal (1-4),Waktu belajar mingguan
3,absences,Numerik,Jumlah ketidakhadiran
4,goout,Ordinal (1-5),Frekuensi keluar dengan teman
5,"Walc, Dalc",Ordinal (1-5),Konsumsi alkohol akhir pekan & hari biasa
6,"Medu, Fedu",Ordinal (0-4),Pendidikan ibu & ayah
7,schoolsup,Biner,Bimbingan belajar dari sekolah
8,higher,Biner,Berniat ke perguruan tinggi
9,"Mjob, Fjob",Nominal,Pekerjaan ibu & ayah


---
## Bab 4 -- Preprocessing: Menyiapkan Data untuk Model

Algoritma ML hanya memahami angka. Kita perlu mengubah variabel kategorik menjadi numerik.

| Jenis | Contoh | Strategi Encoding |
|-------|--------|-------------------|
| **Biner** | schoolsup (yes/no) | Pemetaan langsung 0/1 |
| **Nominal** | Mjob (teacher, health, ...) | One-Hot Encoding |
| **Numerik/Ordinal** | age, studytime, G1 | Dibiarkan apa adanya |

> **Catatan:** Decision Tree tidak memerlukan scaling (normalisasi/standarisasi) karena hanya membandingkan nilai dengan ambang batas.

In [11]:
# === Langkah 1: Encoding Variabel Biner ===

# Memetakan variabel biner yes/no menjadi 1/0
biner = ["schoolsup", "famsup", "paid", "activities",
         "higher", "internet", "romantic", "nursery"]
for kol in biner:
    df[kol] = df[kol].map({"yes": 1, "no": 0})

# Variabel dua-kategori lainnya
df["sex"]     = df["sex"].map({"M": 1, "F": 0})
df["address"] = df["address"].map({"U": 1, "R": 0})  # Urban / Rural
df["famsize"] = df["famsize"].map({"GT3": 1, "LE3": 0})
df["Pstatus"] = df["Pstatus"].map({"T": 1, "A": 0})  # Together / Apart
df["school"]  = df["school"].map({"GP": 1, "MS": 0})

print("Encoding biner selesai.")
print(f"Contoh -- schoolsup: {df['schoolsup'].unique()}")
print(f"Contoh -- sex:       {df['sex'].unique()}")

Encoding biner selesai.
Contoh -- schoolsup: [1 0]
Contoh -- sex:       [0 1]


In [12]:
# === Langkah 2: One-Hot Encoding Variabel Nominal ===

# Variabel nominal multi-kategori (tanpa urutan alami)
nominal = ["Mjob", "Fjob", "reason", "guardian"]
df = pd.get_dummies(df, columns=nominal, drop_first=False)

print("One-Hot Encoding selesai.")
print(f"Dimensi data setelah encoding: {df.shape}")
print(f"\nContoh kolom baru dari Mjob:")
print([col for col in df.columns if col.startswith('Mjob_')])

One-Hot Encoding selesai.
Dimensi data setelah encoding: (395, 47)

Contoh kolom baru dari Mjob:
['Mjob_at_home', 'Mjob_health', 'Mjob_other', 'Mjob_services', 'Mjob_teacher']


In [13]:
# === Langkah 3: Memisahkan Fitur (X) dan Target (y) ===

# PENTING: Buang G3 dari X karena ia adalah sumber target (data leakage!)
# G1 & G2 tetap boleh -- tersedia sebelum ujian akhir
X = df.drop(columns=["G3", "lulus"])
y = df["lulus"]

print(f"Jumlah fitur setelah encoding: {X.shape[1]}")
print(f"Jumlah sampel: {X.shape[0]}")
print(f"\nDaftar fitur:\n{list(X.columns)}")

Jumlah fitur setelah encoding: 45
Jumlah sampel: 395

Daftar fitur:
['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'Mjob_at_home', 'Mjob_health', 'Mjob_other', 'Mjob_services', 'Mjob_teacher', 'Fjob_at_home', 'Fjob_health', 'Fjob_other', 'Fjob_services', 'Fjob_teacher', 'reason_course', 'reason_home', 'reason_other', 'reason_reputation', 'guardian_father', 'guardian_mother', 'guardian_other']


In [14]:
# === Langkah 4: Membagi Data Latih & Uji ===

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,     # 25% untuk data uji
    random_state=42,    # agar hasil dapat direproduksi
    stratify=y          # jaga proporsi kelas tetap sama
)

print(f"Data latih: {X_train.shape[0]} siswa")
print(f"Data uji  : {X_test.shape[0]} siswa")
print(f"\nProporsi Lulus di data latih: {y_train.mean():.3f}")
print(f"Proporsi Lulus di data uji  : {y_test.mean():.3f}")

Data latih: 296 siswa
Data uji  : 99 siswa

Proporsi Lulus di data latih: 0.672
Proporsi Lulus di data uji  : 0.667


---
## Bab 5 -- Membangun & Memvisualisasikan Pohon Keputusan

Kita mulai dari pohon **tanpa batasan** (baseline) -- biarkan algoritma tumbuh sebebas-bebasnya.
Ini menjadi titik acuan untuk mengukur perbaikan di bab selanjutnya.

In [15]:
# === Melatih Pohon Pertama (Baseline -- tanpa batasan) ===

tree_baseline = DecisionTreeClassifier(random_state=42)
tree_baseline.fit(X_train, y_train)

# Mengukur akurasi pada data latih dan data uji
acc_train_base = accuracy_score(y_train, tree_baseline.predict(X_train))
acc_test_base  = accuracy_score(y_test, tree_baseline.predict(X_test))

print(f"Akurasi latih : {acc_train_base:.3f}")
print(f"Akurasi uji   : {acc_test_base:.3f}")
print(f"Kedalaman pohon: {tree_baseline.get_depth()}")
print(f"Jumlah daun   : {tree_baseline.get_n_leaves()}")

print(f"\n⚠️ OVERFITTING TERDETEKSI!")
print(f"Selisih akurasi latih-uji: {(acc_train_base - acc_test_base)*100:.1f}%")
print(f"Akurasi latih {acc_train_base*100:.0f}% (menghafal) vs uji {acc_test_base*100:.1f}% (generalisasi buruk)")

Akurasi latih : 1.000
Akurasi uji   : 0.859
Kedalaman pohon: 7
Jumlah daun   : 21

⚠️ OVERFITTING TERDETEKSI!
Selisih akurasi latih-uji: 14.1%
Akurasi latih 100% (menghafal) vs uji 85.9% (generalisasi buruk)


In [16]:
# === Visualisasi Pohon Baseline ===

plt.figure(figsize=(24, 12))
plot_tree(tree_baseline,
          feature_names=X.columns,
          class_names=["Gagal", "Lulus"],
          filled=True,
          rounded=True,
          fontsize=7)
plt.title("Pohon Keputusan Baseline (Tanpa Pruning) -- Terlalu Kompleks!", fontsize=14)
plt.tight_layout()
plt.show()

print("Pohon ini terlalu lebat -- kita akan memangkasnya di Bab 6.")

Pohon ini terlalu lebat -- kita akan memangkasnya di Bab 6.


In [17]:
# === Pohon Baseline dalam Bentuk Teks ===

aturan_baseline = export_text(tree_baseline, feature_names=list(X.columns))
print("Struktur pohon baseline (teks):")
print(aturan_baseline[:2000])  # Tampilkan sebagian
print("... (terlalu panjang -- akan disederhanakan setelah pruning)")

Struktur pohon baseline (teks):
|--- G2 <= 9.50
|   |--- G2 <= 8.50
|   |   |--- traveltime <= 3.50
|   |   |   |--- traveltime <= 2.50
|   |   |   |   |--- class: 0
|   |   |   |--- traveltime >  2.50
|   |   |   |   |--- goout <= 4.50
|   |   |   |   |   |--- class: 0
|   |   |   |   |--- goout >  4.50
|   |   |   |   |   |--- class: 1
|   |   |--- traveltime >  3.50
|   |   |   |--- sex <= 0.50
|   |   |   |   |--- class: 1
|   |   |   |--- sex >  0.50
|   |   |   |   |--- class: 0
|   |--- G2 >  8.50
|   |   |--- Fjob_other <= 0.50
|   |   |   |--- school <= 0.50
|   |   |   |   |--- address <= 0.50
|   |   |   |   |   |--- class: 1
|   |   |   |   |--- address >  0.50
|   |   |   |   |   |--- class: 0
|   |   |   |--- school >  0.50
|   |   |   |   |--- class: 0
|   |   |--- Fjob_other >  0.50
|   |   |   |--- health <= 4.50
|   |   |   |   |--- famrel <= 4.50
|   |   |   |   |   |--- guardian_father <= 0.50
|   |   |   |   |   |   |--- health <= 1.50
|   |   |   |   |   |   |   |

---
## Bab 6 -- Pruning: Mencegah Overfitting

Pohon yang dibiarkan tumbuh liar akan **menghafal**, bukan belajar.

### Tiga Tuas Pengendali Kompleksitas

| Hyperparameter | Peran | Efek bila dinaikkan |
|----------------|-------|--------------------|
| `max_depth` | Batas kedalaman maksimum | Pohon lebih sederhana |
| `min_samples_split` | Min. sampel untuk split | Split hanya pada kelompok besar |
| `min_samples_leaf` | Min. sampel di setiap daun | Mencegah daun "mikro" |

In [18]:
# === Efek max_depth terhadap Akurasi (Gambar 6.1 di ebook) ===

depths = range(1, 15)
train_accs = []
test_accs = []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, dt.predict(X_train)))
    test_accs.append(accuracy_score(y_test, dt.predict(X_test)))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(depths, train_accs, 'o-', label='Akurasi Latih', linewidth=2)
ax.plot(depths, test_accs, 's--', label='Akurasi Uji', linewidth=2)

best_depth = depths[np.argmax(test_accs)]
ax.axvline(best_depth, color='green', linestyle=':', alpha=0.7,
           label=f'Sweet spot (depth={best_depth})')

ax.set_xlabel('max_depth')
ax.set_ylabel('Akurasi')
ax.set_title('Gambar 6.1 -- Akurasi Latih vs Uji pada Berbagai max_depth')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Sweet spot: max_depth={best_depth} dengan akurasi uji {max(test_accs):.3f}")

Sweet spot: max_depth=3 dengan akurasi uji 0.899


In [19]:
# === Mencari Kombinasi Terbaik dengan GridSearchCV ===

param_grid = {
    "criterion":        ["gini", "entropy"],
    "max_depth":        [3, 4, 5, 6, 8, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf":  [1, 5, 10, 20],
}

# Cari kombinasi terbaik dengan validasi silang 5-lipat
grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,              # 5-fold cross-validation
    scoring="f1",      # optimalkan F1-score
    n_jobs=1,          # gunakan 1 core (stabil di semua environment)
)

grid.fit(X_train, y_train)

print("Parameter terbaik:", grid.best_params_)
print(f"Skor F1 (CV) terbaik: {grid.best_score_:.3f}")

Parameter terbaik: {'criterion': 'gini', 'max_depth': 4, 'min_samples_leaf': 10, 'min_samples_split': 2}
Skor F1 (CV) terbaik: 0.953


In [20]:
# === Model Final Setelah Pruning ===

model = grid.best_estimator_

acc_train_pruned = accuracy_score(y_train, model.predict(X_train))
acc_test_pruned  = accuracy_score(y_test, model.predict(X_test))

print(f"Akurasi uji setelah pruning: {acc_test_pruned:.3f}")
print(f"Kedalaman: {model.get_depth()} | Daun: {model.get_n_leaves()}")

# Perbandingan sebelum vs sesudah pruning
print("\n" + "="*50)
print(f"{'Metrik':<20} {'Sebelum':>10} {'Sesudah':>10}")
print("="*50)
print(f"{'Akurasi latih':<20} {acc_train_base:>10.1%} {acc_train_pruned:>10.1%}")
print(f"{'Akurasi uji':<20} {acc_test_base:>10.1%} {acc_test_pruned:>10.1%}")
print(f"{'Kedalaman':<20} {tree_baseline.get_depth():>10} {model.get_depth():>10}")
print(f"{'Jumlah daun':<20} {tree_baseline.get_n_leaves():>10} {model.get_n_leaves():>10}")
print("="*50)
print("\nPohon lebih sederhana, akurasi uji justru NAIK!")

Akurasi uji setelah pruning: 0.899
Kedalaman: 4 | Daun: 9

Metrik                  Sebelum    Sesudah
Akurasi latih            100.0%      94.3%
Akurasi uji               85.9%      89.9%
Kedalaman                     7          4
Jumlah daun                  21          9

Pohon lebih sederhana, akurasi uji justru NAIK!


---
## Bab 7 -- Evaluasi, Visualisasi & Insight

Model yang baik tidak cukup hanya akurat -- ia harus bisa **dipahami**.

In [21]:
# === Pohon Keputusan Final (Gambar 7.1 di ebook) ===

plt.figure(figsize=(22, 10))
plot_tree(model,
          feature_names=X.columns,
          class_names=["Gagal", "Lulus"],
          filled=True,
          rounded=True,
          fontsize=9)
plt.title("Gambar 7.1 -- Pohon Keputusan Final (Setelah Pruning)", fontsize=14)
plt.tight_layout()
plt.show()

print("Node biru = cenderung Lulus, oranye = cenderung Gagal.")
print("Warna makin pekat = node makin murni.")

Node biru = cenderung Lulus, oranye = cenderung Gagal.
Warna makin pekat = node makin murni.


In [22]:
# === Pohon Final dalam Bentuk Teks ===

aturan_final = export_text(model, feature_names=list(X.columns))
print("Struktur pohon final (setiap baris = satu kondisi split):")
print(aturan_final)

Struktur pohon final (setiap baris = satu kondisi split):
|--- G2 <= 9.50
|   |--- G2 <= 8.50
|   |   |--- Medu <= 1.50
|   |   |   |--- studytime <= 1.50
|   |   |   |   |--- class: 0
|   |   |   |--- studytime >  1.50
|   |   |   |   |--- class: 0
|   |   |--- Medu >  1.50
|   |   |   |--- class: 0
|   |--- G2 >  8.50
|   |   |--- Fjob_other <= 0.50
|   |   |   |--- class: 0
|   |   |--- Fjob_other >  0.50
|   |   |   |--- class: 1
|--- G2 >  9.50
|   |--- G1 <= 10.50
|   |   |--- famsize <= 0.50
|   |   |   |--- class: 1
|   |   |--- famsize >  0.50
|   |   |   |--- age <= 16.50
|   |   |   |   |--- class: 1
|   |   |   |--- age >  16.50
|   |   |   |   |--- class: 1
|   |--- G1 >  10.50
|   |   |--- class: 1



In [23]:
# === Performa Model pada Data Uji ===

y_pred = model.predict(X_test)

print("Classification Report:")
print("="*55)
print(classification_report(y_test, y_pred,
                            target_names=["Gagal", "Lulus"]))

Classification Report:
              precision    recall  f1-score   support

       Gagal       0.83      0.88      0.85        33
       Lulus       0.94      0.91      0.92        66

    accuracy                           0.90        99
   macro avg       0.88      0.89      0.89        99
weighted avg       0.90      0.90      0.90        99



In [24]:
# === Confusion Matrix (Gambar 7.2 di ebook) ===

fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=["Gagal", "Lulus"])
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title("Gambar 7.2 -- Confusion Matrix pada Data Uji")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"\nDari {tn + fn} siswa yang benar-benar Gagal:")
print(f"  - {tn} terdeteksi dengan benar (True Negative)")
print(f"  - {fn} lolos dari deteksi (False Negative)")
print(f"\nRecall kelas Gagal: {tn/(tn+fp)*100:.1f}% -- penting untuk peringatan dini!")


Dari 35 siswa yang benar-benar Gagal:
  - 29 terdeteksi dengan benar (True Negative)
  - 6 lolos dari deteksi (False Negative)

Recall kelas Gagal: 87.9% -- penting untuk peringatan dini!


In [25]:
# === Feature Importance -- Model Peringatan Dini (Gambar 7.3 kiri) ===

importances = model.feature_importances_
feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=False)
top_features = feat_imp[feat_imp > 0]

fig, ax = plt.subplots(figsize=(8, 5))
top_features.plot(kind='barh', ax=ax, color='steelblue', edgecolor='black')
ax.set_xlabel('Kepentingan Fitur (Feature Importance)')
ax.set_title('Gambar 7.3 (Kiri) -- Feature Importance: Model Peringatan Dini')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print("Fitur terpenting:")
for f, v in top_features.items():
    print(f"  {f}: {v:.1%}")

Fitur terpenting:
  G2: 90.2%
  Fjob_other: 5.3%
  G1: 1.6%
  famsize: 1.2%
  age: 1.0%
  Medu: 0.6%
  studytime: 0.1%


In [26]:
# === Model "Latar Belakang Saja" -- Tanpa Nilai Interim (Gambar 7.3 kanan) ===
# Tujuan: menyingkap faktor sosial yang DAPAT DIINTERVENSI

X_sosial = X.drop(columns=["G1", "G2"])
X_sosial_train = X_train.drop(columns=["G1", "G2"])
X_sosial_test  = X_test.drop(columns=["G1", "G2"])

tree_sosial = DecisionTreeClassifier(
    max_depth=4, min_samples_leaf=10, random_state=42
)
tree_sosial.fit(X_sosial_train, y_train)

acc_sosial = accuracy_score(y_test, tree_sosial.predict(X_sosial_test))
print(f"Akurasi model latar-belakang (tanpa G1, G2): {acc_sosial:.3f}")
print("(Lebih rendah, tapi insight-nya lebih dapat ditindaklanjuti)\n")

# Feature importance model sosial
imp_sosial = pd.Series(
    tree_sosial.feature_importances_, index=X_sosial.columns
).sort_values(ascending=False)
top_sosial = imp_sosial[imp_sosial > 0]

fig, ax = plt.subplots(figsize=(8, 5))
top_sosial.plot(kind='barh', ax=ax, color='coral', edgecolor='black')
ax.set_xlabel('Kepentingan Fitur')
ax.set_title('Gambar 7.3 (Kanan) -- Feature Importance: Model Latar Belakang')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print("Faktor sosial terpenting:")
for f, v in top_sosial.head(8).items():
    print(f"  {f}: {v:.1%}")

Akurasi model latar-belakang (tanpa G1, G2): 0.636
(Lebih rendah, tapi insight-nya lebih dapat ditindaklanjuti)

Faktor sosial terpenting:
  failures: 53.7%
  absences: 9.4%
  Walc: 9.2%
  goout: 8.3%
  Dalc: 5.2%
  age: 5.0%
  schoolsup: 3.8%
  guardian_father: 3.2%


In [27]:
# === Learning Curve (Gambar 7.5 di ebook) ===

train_sizes, train_scores, val_scores = learning_curve(
    model, X_train, y_train,
    cv=5,
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='accuracy',
    n_jobs=1
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(train_sizes, train_scores.mean(axis=1), 'o-', label='Skor Latih', linewidth=2)
ax.plot(train_sizes, val_scores.mean(axis=1), 's--', label='Skor Validasi', linewidth=2)
ax.fill_between(train_sizes,
                train_scores.mean(axis=1) - train_scores.std(axis=1),
                train_scores.mean(axis=1) + train_scores.std(axis=1), alpha=0.1)
ax.fill_between(train_sizes,
                val_scores.mean(axis=1) - val_scores.std(axis=1),
                val_scores.mean(axis=1) + val_scores.std(axis=1), alpha=0.1)
ax.set_xlabel('Jumlah Data Latih')
ax.set_ylabel('Akurasi')
ax.set_title('Gambar 7.5 -- Learning Curve')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Kedua kurva mendekat di skor tinggi -- model sehat, tidak overfit berat.")

Kedua kurva mendekat di skor tinggi -- model sehat, tidak overfit berat.


---
## Bab 8 -- Interpretasi Aturan untuk Intervensi Pendidikan

Sebuah model hanya bernilai bila **menggerakkan tindakan**.

In [28]:
# === Tabel Ringkasan Aturan Keputusan ===

print("ATURAN KEPUTUSAN DARI POHON FINAL")
print("=" * 70)
print()
print(export_text(model, feature_names=list(X.columns)))

print("=" * 70)
print("\nATURAN EMAS:")
print("Split akar G2 <= 9.5 adalah garis pemisah utama.")
print("Siswa dengan nilai periode 2 di bawah ambang ini")
print("langsung masuk 'zona pantau' -- pemicu peringatan dini terkuat.")

ATURAN KEPUTUSAN DARI POHON FINAL

|--- G2 <= 9.50
|   |--- G2 <= 8.50
|   |   |--- Medu <= 1.50
|   |   |   |--- studytime <= 1.50
|   |   |   |   |--- class: 0
|   |   |   |--- studytime >  1.50
|   |   |   |   |--- class: 0
|   |   |--- Medu >  1.50
|   |   |   |--- class: 0
|   |--- G2 >  8.50
|   |   |--- Fjob_other <= 0.50
|   |   |   |--- class: 0
|   |   |--- Fjob_other >  0.50
|   |   |   |--- class: 1
|--- G2 >  9.50
|   |--- G1 <= 10.50
|   |   |--- famsize <= 0.50
|   |   |   |--- class: 1
|   |   |--- famsize >  0.50
|   |   |   |--- age <= 16.50
|   |   |   |   |--- class: 1
|   |   |   |--- age >  16.50
|   |   |   |   |--- class: 1
|   |--- G1 >  10.50
|   |   |--- class: 1


ATURAN EMAS:
Split akar G2 <= 9.5 adalah garis pemisah utama.
Siswa dengan nilai periode 2 di bawah ambang ini
langsung masuk 'zona pantau' -- pemicu peringatan dini terkuat.


In [29]:
# === Peta Intervensi: Dari Aturan ke Aksi ===

intervensi = pd.DataFrame({
    'Temuan Model': [
        'G2 <= 9.5',
        'failures > 0',
        'absences tinggi',
        'goout tinggi',
        'schoolsup = yes'
    ],
    'Interpretasi': [
        'Risiko kegagalan tinggi pada ujian akhir',
        'Pernah gagal kelas -- prediktor risiko terkuat',
        'Ketidakhadiran berkorelasi dengan kegagalan',
        'Pola sosial berisiko mengganggu fokus belajar',
        'Dukungan sekolah berdampak pada kelulusan'
    ],
    'Aksi Intervensi': [
        'Kelas remedial & pendampingan intensif sebelum ujian akhir',
        'Mentoring personal & pemantauan kemajuan mingguan',
        'Sistem pemantauan kehadiran & komunikasi dini dengan orang tua',
        'Konseling manajemen waktu & kegiatan terstruktur',
        'Perluas akses program bimbingan belajar resmi'
    ]
})

print("PETA INTERVENSI PENDIDIKAN")
print("=" * 90)
for _, row in intervensi.iterrows():
    print(f"\n  Temuan      : {row['Temuan Model']}")
    print(f"  Interpretasi: {row['Interpretasi']}")
    print(f"  Aksi        : {row['Aksi Intervensi']}")
    print("  " + "-"*60)

print("\n[!] Prediksi bukan vonis. Gunakan model untuk membuka peluang bantuan,")
print("    bukan untuk melabeli atau mengucilkan siswa.")

PETA INTERVENSI PENDIDIKAN

  Temuan      : G2 <= 9.5
  Interpretasi: Risiko kegagalan tinggi pada ujian akhir
  Aksi        : Kelas remedial & pendampingan intensif sebelum ujian akhir
  ------------------------------------------------------------

  Temuan      : failures > 0
  Interpretasi: Pernah gagal kelas -- prediktor risiko terkuat
  Aksi        : Mentoring personal & pemantauan kemajuan mingguan
  ------------------------------------------------------------

  Temuan      : absences tinggi
  Interpretasi: Ketidakhadiran berkorelasi dengan kegagalan
  Aksi        : Sistem pemantauan kehadiran & komunikasi dini dengan orang tua
  ------------------------------------------------------------

  Temuan      : goout tinggi
  Interpretasi: Pola sosial berisiko mengganggu fokus belajar
  Aksi        : Konseling manajemen waktu & kegiatan terstruktur
  ------------------------------------------------------------

  Temuan      : schoolsup = yes
  Interpretasi: Dukungan sekolah berdampa

---
## Ekspor Model

Sebelum menguji deployment, kita simpan model dan daftar fitur ke file `.pkl` agar bisa dimuat kembali **tanpa perlu melatih ulang**. Ini mensimulasikan alur deployment nyata: *train once, predict anywhere*.

In [30]:
# === Ekspor Model & Daftar Fitur ke File .pkl ===

import joblib

joblib.dump(model, 'model_decision_tree_siswa.pkl')
joblib.dump(list(X.columns), 'fitur_kolom.pkl')

print("Model tersimpan sebagai 'model_decision_tree_siswa.pkl'")
print("Daftar fitur tersimpan sebagai 'fitur_kolom.pkl'")
print(f"\nModel: {type(model).__name__}")
print(f"Kedalaman: {model.get_depth()} | Daun: {model.get_n_leaves()}")
print(f"Jumlah fitur: {len(X.columns)}")

Model tersimpan sebagai 'model_decision_tree_siswa.pkl'
Daftar fitur tersimpan sebagai 'fitur_kolom.pkl'

Model: DecisionTreeClassifier
Kedalaman: 4 | Daun: 9
Jumlah fitur: 45


---
## 9. Uji Deployment: Prediksi Siswa Baru

Bagian ini mensimulasikan penggunaan model **di dunia nyata** -- model dimuat dari file `.pkl` yang sudah diekspor, bukan dari variabel di memori. Ini membuktikan bahwa model benar-benar portable dan siap dipakai di sistem lain.

Alur:
1. Muat model & daftar fitur dari file `.pkl`
2. Buat fungsi prediksi berbasis model yang dimuat
3. Uji dengan beberapa skenario siswa baru
4. Prediksi batch (massal)

In [31]:
# === Muat Model dari File .pkl (Simulasi Deployment) ===

import joblib

model_loaded = joblib.load('model_decision_tree_siswa.pkl')
fitur_loaded = joblib.load('fitur_kolom.pkl')

print("Model berhasil dimuat dari 'model_decision_tree_siswa.pkl'")
print(f"Tipe model  : {type(model_loaded).__name__}")
print(f"Jumlah fitur: {len(fitur_loaded)}")
print(f"Kedalaman   : {model_loaded.get_depth()} | Daun: {model_loaded.get_n_leaves()}")


# === Fungsi Prediksi Berbasis Model yang Dimuat ===

def prediksi_siswa(data_siswa: dict):
    """
    Menerima dictionary data siswa, mengembalikan prediksi dan probabilitas.
    Menggunakan model dan daftar fitur yang dimuat dari file .pkl.
    """
    input_df = pd.DataFrame([data_siswa])

    # Pastikan semua kolom ada (isi 0 untuk kolom one-hot yang tidak diberikan)
    for col in fitur_loaded:
        if col not in input_df.columns:
            input_df[col] = 0
    input_df = input_df[fitur_loaded]

    prediksi = model_loaded.predict(input_df)[0]
    probabilitas = model_loaded.predict_proba(input_df)[0]

    status = "LULUS" if prediksi == 1 else "GAGAL"
    confidence = max(probabilitas) * 100

    return {
        'prediksi': status,
        'confidence': f"{confidence:.1f}%",
        'prob_gagal': f"{probabilitas[0]:.1%}",
        'prob_lulus': f"{probabilitas[1]:.1%}"
    }

print("\nFungsi prediksi_siswa() siap -- menggunakan model dari file .pkl")

Model berhasil dimuat dari 'model_decision_tree_siswa.pkl'
Tipe model  : DecisionTreeClassifier
Jumlah fitur: 45
Kedalaman   : 4 | Daun: 9

Fungsi prediksi_siswa() siap -- menggunakan model dari file .pkl


In [32]:
# === Template Data Siswa Baru ===
# Gunakan template ini sebagai panduan mengisi data

template_siswa = {
    # === Identitas ===
    'school': 1,        # 1=GP, 0=MS
    'sex': 1,           # 1=Male, 0=Female
    'age': 17,          # Usia siswa
    'address': 1,       # 1=Urban, 0=Rural
    'famsize': 1,       # 1=GT3 (>3 anggota), 0=LE3
    'Pstatus': 1,       # 1=Together, 0=Apart

    # === Pendidikan Orang Tua ===
    'Medu': 3,          # Pendidikan ibu (0-4)
    'Fedu': 2,          # Pendidikan ayah (0-4)

    # === Akademik ===
    'traveltime': 1,    # Waktu tempuh ke sekolah (1-4)
    'studytime': 2,     # Waktu belajar mingguan (1-4)
    'failures': 0,      # Riwayat gagal kelas (0-3)
    'schoolsup': 0,     # Bimbingan dari sekolah (1=yes)
    'famsup': 1,        # Dukungan keluarga (1=yes)
    'paid': 0,          # Les tambahan (1=yes)
    'activities': 1,    # Kegiatan ekstra (1=yes)
    'nursery': 1,       # Pernah TK (1=yes)
    'higher': 1,        # Ingin kuliah (1=yes)
    'internet': 1,      # Akses internet (1=yes)
    'romantic': 0,      # Pacaran (1=yes)

    # === Nilai Interim ===
    'G1': 12,           # Nilai periode 1 (0-20)
    'G2': 11,           # Nilai periode 2 (0-20)

    # === Sosial ===
    'famrel': 4,        # Hubungan keluarga (1-5)
    'freetime': 3,      # Waktu luang (1-5)
    'goout': 3,         # Keluar dengan teman (1-5)
    'Dalc': 1,          # Alkohol hari biasa (1-5)
    'Walc': 2,          # Alkohol akhir pekan (1-5)
    'health': 4,        # Kesehatan (1-5)
    'absences': 2,      # Ketidakhadiran

    # === One-Hot: Pekerjaan Ibu (pilih salah satu = 1) ===
    'Mjob_at_home': 0, 'Mjob_health': 0, 'Mjob_other': 0,
    'Mjob_services': 0, 'Mjob_teacher': 1,

    # === One-Hot: Pekerjaan Ayah (pilih salah satu = 1) ===
    'Fjob_at_home': 0, 'Fjob_health': 0, 'Fjob_other': 0,
    'Fjob_services': 1, 'Fjob_teacher': 0,

    # === One-Hot: Alasan Pilih Sekolah ===
    'reason_course': 1, 'reason_home': 0,
    'reason_other': 0, 'reason_reputation': 0,

    # === One-Hot: Wali ===
    'guardian_father': 0, 'guardian_mother': 1, 'guardian_other': 0,
}

print("Template data siswa tersedia.")
print(f"Jumlah field: {len(template_siswa)}")

Template data siswa tersedia.
Jumlah field: 45


In [33]:
# === Uji Skenario: 4 Profil Siswa Berbeda ===

skenario = {
    "Siswa A (Berprestasi)": {
        **template_siswa,
        'G1': 15, 'G2': 16, 'studytime': 4, 'failures': 0,
        'absences': 1, 'goout': 2, 'higher': 1
    },
    "Siswa B (Berisiko Tinggi)": {
        **template_siswa,
        'G1': 7, 'G2': 6, 'studytime': 1, 'failures': 2,
        'absences': 15, 'goout': 5, 'Walc': 4, 'higher': 0
    },
    "Siswa C (Di Ambang Batas)": {
        **template_siswa,
        'G1': 10, 'G2': 9, 'studytime': 2, 'failures': 1,
        'absences': 8, 'goout': 3, 'schoolsup': 1
    },
    "Siswa D (Sosial Tinggi, Nilai Cukup)": {
        **template_siswa,
        'G1': 11, 'G2': 10, 'studytime': 2, 'failures': 0,
        'absences': 5, 'goout': 5, 'Walc': 3, 'romantic': 1
    }
}

print("HASIL PREDIKSI SISWA BARU")
print("=" * 65)

for nama, data in skenario.items():
    hasil = prediksi_siswa(data)
    emoji = "[V]" if hasil['prediksi'] == 'LULUS' else "[X]"
    print(f"\n{emoji} {nama}")
    print(f"   G1={data['G1']}, G2={data['G2']}, failures={data['failures']}, "
          f"absences={data['absences']}, goout={data['goout']}")
    print(f"   Prediksi  : {hasil['prediksi']} (confidence: {hasil['confidence']})")
    print(f"   P(Gagal)  : {hasil['prob_gagal']}")
    print(f"   P(Lulus)  : {hasil['prob_lulus']}")

print("\n" + "=" * 65)

HASIL PREDIKSI SISWA BARU

[V] Siswa A (Berprestasi)
   G1=15, G2=16, failures=0, absences=1, goout=2
   Prediksi  : LULUS (confidence: 100.0%)
   P(Gagal)  : 0.0%
   P(Lulus)  : 100.0%

[X] Siswa B (Berisiko Tinggi)
   G1=7, G2=6, failures=2, absences=15, goout=5
   Prediksi  : GAGAL (confidence: 100.0%)
   P(Gagal)  : 100.0%
   P(Lulus)  : 0.0%

[X] Siswa C (Di Ambang Batas)
   G1=10, G2=9, failures=1, absences=8, goout=3
   Prediksi  : GAGAL (confidence: 94.7%)
   P(Gagal)  : 94.7%
   P(Lulus)  : 5.3%

[V] Siswa D (Sosial Tinggi, Nilai Cukup)
   G1=11, G2=10, failures=0, absences=5, goout=5
   Prediksi  : LULUS (confidence: 100.0%)
   P(Gagal)  : 0.0%
   P(Lulus)  : 100.0%



In [34]:
# === Input Manual: Masukkan Data Siswa Sendiri ===

print("PREDIKSI MANUAL -- Ubah nilai di bawah sesuai data siswa Anda")
print("=" * 60)

siswa_baru = template_siswa.copy()

# ======= UBAH DATA DI SINI =======
siswa_baru['G1'] = 8          # Nilai periode 1 (0-20)
siswa_baru['G2'] = 9          # Nilai periode 2 (0-20)
siswa_baru['failures'] = 1    # Riwayat gagal (0-3)
siswa_baru['studytime'] = 2   # Waktu belajar (1-4)
siswa_baru['absences'] = 10   # Ketidakhadiran
siswa_baru['goout'] = 4       # Keluar dgn teman (1-5)
siswa_baru['Medu'] = 2        # Pendidikan ibu (0-4)
siswa_baru['age'] = 17        # Usia
# ==================================

hasil = prediksi_siswa(siswa_baru)

print(f"\nHASIL PREDIKSI:")
print(f"  Status    : {hasil['prediksi']}")
print(f"  Confidence: {hasil['confidence']}")
print(f"  P(Gagal)  : {hasil['prob_gagal']}")
print(f"  P(Lulus)  : {hasil['prob_lulus']}")

if hasil['prediksi'] == 'GAGAL':
    print("\n  >> REKOMENDASI INTERVENSI:")
    if siswa_baru['G2'] <= 9.5:
        print("     - Kelas remedial & pendampingan belajar intensif")
    if siswa_baru['failures'] > 0:
        print("     - Mentoring personal & pemantauan mingguan")
    if siswa_baru['absences'] > 6:
        print("     - Pemantauan kehadiran & komunikasi dengan orang tua")
    if siswa_baru['goout'] >= 4:
        print("     - Konseling manajemen waktu")
else:
    print("\n  >> Siswa dalam zona aman. Pertahankan performa!")

PREDIKSI MANUAL -- Ubah nilai di bawah sesuai data siswa Anda

HASIL PREDIKSI:
  Status    : GAGAL
  Confidence: 94.7%
  P(Gagal)  : 94.7%
  P(Lulus)  : 5.3%

  >> REKOMENDASI INTERVENSI:
     - Kelas remedial & pendampingan belajar intensif
     - Mentoring personal & pemantauan mingguan
     - Pemantauan kehadiran & komunikasi dengan orang tua
     - Konseling manajemen waktu


In [35]:
# === Simulasi Batch: Prediksi Banyak Siswa Sekaligus ===

data_batch = pd.DataFrame([
    {**template_siswa, 'G1': 14, 'G2': 15, 'failures': 0, 'absences': 2},
    {**template_siswa, 'G1': 6,  'G2': 5,  'failures': 2, 'absences': 20},
    {**template_siswa, 'G1': 10, 'G2': 10, 'failures': 0, 'absences': 4},
    {**template_siswa, 'G1': 8,  'G2': 7,  'failures': 1, 'absences': 12},
    {**template_siswa, 'G1': 12, 'G2': 13, 'failures': 0, 'absences': 0},
])

for col in fitur_loaded:
    if col not in data_batch.columns:
        data_batch[col] = 0
data_batch_X = data_batch[fitur_loaded]

prediksi_batch = model_loaded.predict(data_batch_X)
proba_batch = model_loaded.predict_proba(data_batch_X)

hasil_batch = pd.DataFrame({
    'Siswa': [f'Siswa {i+1}' for i in range(len(data_batch))],
    'G1': data_batch['G1'].values,
    'G2': data_batch['G2'].values,
    'Failures': data_batch['failures'].values,
    'Absences': data_batch['absences'].values,
    'Prediksi': ['LULUS' if p == 1 else 'GAGAL' for p in prediksi_batch],
    'Confidence': [f"{max(p)*100:.1f}%" for p in proba_batch],
})

print("SIMULASI BATCH -- Prediksi 5 Siswa (menggunakan model dari file .pkl)")
print("=" * 65)
print(hasil_batch.to_string(index=False))
print("\nModel siap digunakan untuk prediksi massal (deployment ready).")

SIMULASI BATCH -- Prediksi 5 Siswa (menggunakan model dari file .pkl)
  Siswa  G1  G2  Failures  Absences Prediksi Confidence
Siswa 1  14  15         0         2    LULUS     100.0%
Siswa 2   6   5         2        20    GAGAL     100.0%
Siswa 3  10  10         0         4    LULUS      72.7%
Siswa 4   8   7         1        12    GAGAL     100.0%
Siswa 5  12  13         0         0    LULUS     100.0%

Model siap digunakan untuk prediksi massal (deployment ready).


---
## Identitas Ebook

| | |
|---|---|
| **Judul** | Prediksi Nilai Siswa dengan Decision Tree |
| **Seri** | Praktik Data Science -- B5 (Tier Pemula) |
| **Penerbit** | Menata Data Lab |
| **Edisi** | Pertama, 2026 |
| **Tagline** | *Boost Your Insight* -- belajar data science dengan praktik nyata |

---

### Dapatkan Ebook Lengkap & Seri Lainnya

Notebook ini adalah **pendamping praktik** dari ebook yang membahas teori, penjelasan baris per baris, tips, peringatan, dan latihan mandiri secara lengkap.

**Dapatkan ebook lengkap dan seri lainnya di:**

| Platform | Link |
|----------|------|
| Semua produk & link | [lynk.id/menatadata](https://lynk.id/menatadata) |
| Instagram | [@menatadata.lab](https://instagram.com/menatadata.lab) |
| TikTok | [@menata.data](https://tiktok.com/@menata.data) |
| GitHub (Kode) | [github.com/DwiAsana99/prediksi-nilai-siswa-dengan-decision-tree](https://github.com/DwiAsana99/prediksi-nilai-siswa-dengan-decision-tree) |

---

*Seri Praktik Data Science -- satu model, satu bidang, satu dataset, satu insight.*

**Tier Pemula**: Regresi, Klasifikasi, Clustering, Decision Tree | **Tier Menengah**: Random Forest, XGBoost, ARIMA, LightGBM | **Tier Mahir**: CNN, Transfer Learning, LSTM, IndoBERT

---

*Copyright 2026 Menata Data Lab -- Seri Praktik Data Science -- B5 -- Edisi Pertama*